# Notebook 05 (Oscar): MLP Training -- Exp 3 and Exp 4

**Exp 3:** Delta Residue + Full Wildtype -- `concat(delta_residue[mut_pos], mean_pool(wt_sequence))`  
**Exp 4:** Delta Sequence -- `mean_pool(mutant_seq) - mean_pool(wt_seq)` (cached)

Both experiments run for both ESM-2 and AbLang2. Training uses MSE loss.
Evaluation metric: Spearman correlation per dataset and aggregate (excluding HER2 separately).
All runs logged to W&B.

## Setup

In [ ]:
import subprocess, os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL = 'https://github.com/Aaron1776/antibody-property-prediction.git'
    REPO_DIR = '/content/antibody-property-prediction'
    BRANCH   = 'implementation'

    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
else:
    REPO_DIR = str(Path('..').resolve())
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

print(f"Environment: {'Colab' if IN_COLAB else 'local'}")
print(f"Repo: {REPO_DIR}")

Detects whether running on Colab or locally. On Colab, mounts Drive and clones
(or pulls) the repo. Locally, resolves repo root from the notebook's location.

Expected output: `Environment: local` (or `Colab`) and the resolved repo path.

In [ ]:
from src.config import DRIVE_ROOT, EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR

for d in [EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Drive root:      {DRIVE_ROOT}")
print(f"Embedding dir:   {EMBEDDING_DIR}")
print(f"Checkpoint dir:  {CHECKPOINT_DIR}")
print("Paths set.")

`src/config.py` resolves `DRIVE_ROOT` automatically across Colab and local.
No per-collaborator edits needed. Checkpoints are saved to Drive so training
can be resumed if Colab disconnects.

In [ ]:
if IN_COLAB:
    subprocess.run(['apt-get', 'install', '-y', 'hmmer'], check=True)
    subprocess.run(['pip', 'install', '-q', '--upgrade', 'ipython'], check=True)
    subprocess.run(['pip', 'install', '-q', 'fair-esm', 'ablang2', 'anarci', 'wandb',
                    'scikit-learn'], check=True)
else:
    print("Local run -- installation skipped.")

In [ ]:
%load_ext autoreload
%autoreload 2

if IN_COLAB:
    subprocess.run(
        ['find', REPO_DIR, '-type', 'd', '-name', '__pycache__', '-exec', 'rm', '-rf', '{}', '+'],
        capture_output=True,
    )

print("Autoreload enabled.")

In [ ]:
import torch
from src.config import DEVICE

print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif DEVICE == 'mps':
    print("Apple MPS -- Apple Silicon unified memory")

Device check. Training runs on GPU (Colab) or MPS (local Apple Silicon).
Expected: `cuda` on Colab T4/A100, `mps` on Mac.

## Imports

In [ ]:
import numpy as np
import pandas as pd
import wandb
from torch.utils.data import DataLoader, Subset

from src.config import DATA_DIR, EMBEDDING_DIR, DEVICE
from src.data.abagym import load_abagym_antibody
from src.data.datasets import AbAgymDataset, EmbeddingStrategy
from src.data.splits import make_stratified_splits
from src.models.mlp import MLP
from src.training.trainer import TrainConfig, train_abagym, evaluate_abagym

print("Imports OK.")

## Data Loading and Splits

In [ ]:
df = load_abagym_antibody(DATA_DIR / 'abagym_antibody.csv')

train_idx, val_idx, test_idx = make_stratified_splits(
    df, val_frac=0.1, test_frac=0.1, random_state=42
)

print(f"Total:  {len(df)}")
print(f"Train:  {len(train_idx)} ({100*len(train_idx)/len(df):.1f}%)")
print(f"Val:    {len(val_idx)} ({100*len(val_idx)/len(df):.1f}%)")
print(f"Test:   {len(test_idx)} ({100*len(test_idx)/len(df):.1f}%)")
print()

# Verify all datasets are represented in each split
for split_name, idx in [('Train', train_idx), ('Val', val_idx), ('Test', test_idx)]:
    counts = df.iloc[idx]['DMS_name'].value_counts().to_dict()
    print(f"{split_name}: {counts}")

Loads the AbAgym metadata CSV and creates the stratified 80/10/10 split.

The split is stratified within each antibody dataset separately, then pooled.
This ensures all 5 antibodies are represented in train, val, and test.
The `random_state=42` is fixed -- Lucas uses the same value in NB05_lucas.ipynb
to guarantee identical splits across both notebooks.

Record confirmed split sizes here after running.

## Experiment 4: Delta Sequence

**Input:** `mean_pool(mutant_seq) - mean_pool(wt_seq)`  
**Dims:** ESM-2 = 2560, AbLang2 = 960  
**Owner:** Oscar

The simplest and most direct embedding strategy. The delta sequence vector captures
the antibody-wide shift in embedding space caused by the mutation. From NB04 EDA,
the L2 norm of this vector is already weakly predictive of mutation effect
(Spearman r=0.083 ESM-2, r=0.216 AbLang2) without any supervised training.
A trained MLP operating on the full 2560/960-dim vector has access to directional
information that the norm discards, so performance should improve substantially.

In [ ]:
# Build datasets for both models -- Exp 4 (DELTA_SEQUENCE)
for model_name in ('esm2', 'ablang2'):
    ds_full = AbAgymDataset(
        antibody_df=df,
        embedding_dir=EMBEDDING_DIR,
        strategy=EmbeddingStrategy.DELTA_SEQUENCE,
        model_name=model_name,
    )
    x0, y0, meta0 = ds_full[0]
    print(f"{model_name} DELTA_SEQUENCE: input_dim={x0.shape[0]}, label={y0:.4f}, region={meta0['region']}")

Sanity check: verifies that the dataset loads correctly and the input dimension
matches expectations (ESM-2=2560, AbLang2=960).

Record confirmed output here after running.

## Experiment 3: Delta Residue + Full Wildtype

**Input:** `concat(delta_residue[mut_pos], mean_pool(wt_sequence))`  
**Dims:** ESM-2 = 1280 + 2560 = 3840, AbLang2 = 480 + 960 = 1440  
**Owner:** Oscar

Combines the local mutation signal (per-token delta at the mutation site) with
global antibody context (wildtype sequence embedding). The rationale: the per-token
delta alone tells you how much the mutation changed that position, but the wildtype
context tells the MLP what kind of antibody this is (CDR vs FR context, scaffold
type, chain identity). Together they provide both local and global information.

The wildtype embedding is shared across all mutations of the same antibody --
the dataset class handles the index lookup internally.

In [ ]:
# Build datasets for both models -- Exp 3 (DELTA_RESIDUE_PLUS_WILD)
for model_name in ('esm2', 'ablang2'):
    ds_full = AbAgymDataset(
        antibody_df=df,
        embedding_dir=EMBEDDING_DIR,
        strategy=EmbeddingStrategy.DELTA_RESIDUE_PLUS_WILD,
        model_name=model_name,
    )
    x0, y0, meta0 = ds_full[0]
    print(f"{model_name} DELTA_RESIDUE_PLUS_WILD: input_dim={x0.shape[0]}, label={y0:.4f}, region={meta0['region']}")

Sanity check: verifies input dimensions for Exp 3
(ESM-2=3840, AbLang2=1440) and that the wildtype index lookup works.

Record confirmed output here after running.